# COVID-19 Death Risk Analysis

This notebook analyzes death outcomes in the CDC COVID-19 Case Surveillance dataset.

The goal is to measure death rates across demographic groups to identify which populations experienced the highest mortality risk.

Death rate is defined as:

deaths / cases with known death status

Only records with death values of "Yes" or "No" are included in the denominator.

In [ ]:
import pandas as pd

df_clean = pd.read_parquet('../data/processed/cdc_clean_1M.parquet')

In [ ]:
df_known_deaths = df_clean[df_clean['death_yn'].isin(['Yes', 'No'])].copy()

df_known_deaths['age_group'] = (
    df_known_deaths['age_group']
    .str.replace(' Years', '', case = False, regex = False)
    .str.replace(' ', '', regex = False)
)

df_known_deaths = df_known_deaths[~df_known_deaths['age_group'].isin(['Missing', 'Unknown'])]

df_known_deaths.head()

In [ ]:
age_order = [
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70-79"
]

death_cases = (
    df_known_deaths[df_known_deaths['death_yn'] == 'Yes'].
    groupby('age_group').
    size()
)

known_cases = (
    df_known_deaths.
    groupby('age_group').
    size()
)


In [ ]:
death_summary = pd.DataFrame({
    'known_cases': known_cases,
    'death_cases': death_cases
})

death_summary['death_rate'] = (
    (death_summary['death_cases']/
     death_summary['known_cases'])*100
)

death_summary = death_summary.reindex(age_order)
death_summary = death_summary.fillna(0)
death_summary['death_rate'] = death_summary['death_rate'].round(2)

In [ ]:
death_summary

In [ ]:
death_summary.to_csv('../outputs/death_by_age.csv')

In [ ]:
import matplotlib.pyplot as plt

plot_df = death_summary[death_summary["known_cases"] > 0]

plt.figure(figsize=(10,6))

plt.bar(
    plot_df.index,
    plot_df["death_rate"]
)

plt.xlabel("Age Group")
plt.ylabel("Death Rate (%)")
plt.ylim(0,3)

plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.title("Death Rate by Age Group (CDC COVID-19 Case Surveillance Sample)")

plt.tight_layout()

plt.savefig("../outputs/death_by_age.png")
plt.show()